# AgroMind Universal — Model Training
**GPU Required**: Runtime → Change runtime type → T4 GPU

This notebook trains two models:
1. EfficientNet-B0 on PlantVillage (38-class disease classifier)
2. MobileNetV3-Small binary plant/non-plant validator

**Expected time**: ~3 hours on Colab T4 GPU
**Output**: `plant_classifier.pth` and `plant_validator.pth`

In [ ]:
# CELL 1: Check GPU
import torch
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NO GPU — change runtime!')
print('CUDA:', torch.version.cuda)
assert torch.cuda.is_available(), 'Please enable GPU: Runtime → Change runtime type → T4'

In [ ]:
# CELL 2: Install dependencies
!pip install timm scikit-learn kaggle -q

In [ ]:
# CELL 3: Download PlantVillage dataset
# Option A: From Kaggle (recommended)
import os
from google.colab import files

print('Upload your kaggle.json API key file:')
files.upload()
os.makedirs('/root/.kaggle', exist_ok=True)
!cp kaggle.json /root/.kaggle/
!chmod 600 /root/.kaggle/kaggle.json
!kaggle datasets download -d emmarex/plantdisease
!unzip -q plantdisease.zip
!ls PlantVillage/ | head -5
print('Dataset ready!')

In [ ]:
# CELL 3B: Alternative — If Kaggle not available
# Download from direct link
!pip install gdown -q
import gdown
# PlantVillage backup from Google Drive
gdown.download('https://drive.google.com/uc?id=0B_voCy5O5sXMTFByemoxQndGQ0k', 'plantvillage.zip', quiet=False)
!unzip -q plantvillage.zip
print('Downloaded from backup source')

In [ ]:
# CELL 4: Verify dataset
import os
from pathlib import Path

pv = Path('PlantVillage')
classes = [d.name for d in pv.iterdir() if d.is_dir()]
total = sum(len(list(d.glob('*.jpg'))) + len(list(d.glob('*.JPG'))) for d in pv.iterdir() if d.is_dir())
print(f'Classes: {len(classes)}')
print(f'Total images: {total}')
print('First 5 classes:', classes[:5])

In [ ]:
# CELL 5: Clone training code
!git clone https://github.com/YOUR_USERNAME/agromind-universal.git
%cd agromind-universal

# OR: Upload files manually
# from google.colab import files
# files.upload()  # upload training/train_plant_classifier.py

In [ ]:
# CELL 6: Train EfficientNet-B0 Classifier
# Expected: ~2.5 hours on T4, ~87% test accuracy
import subprocess
result = subprocess.run(['python', 'training/train_plant_classifier.py'], capture_output=True, text=True)
print(result.stdout[-3000:])  # last 3000 chars
if result.returncode != 0:
    print('ERRORS:', result.stderr[-1000:])

In [ ]:
# CELL 7: View training results
import json
with open('models/plant_metrics.json') as f:
    metrics = json.load(f)
print('=== PLANT CLASSIFIER RESULTS ===')
print(f'Model: {metrics["model"]}')
print(f'Dataset: {metrics["dataset"]} ({metrics["num_samples"]} images)')
print(f'Test Accuracy: {metrics["test_metrics"]["accuracy"]:.4f} ({metrics["test_metrics"]["accuracy"]*100:.1f}%)')
print(f'Test Macro F1: {metrics["test_metrics"]["macro_f1"]:.4f}')
print(f'Test Precision: {metrics["test_metrics"]["macro_precision"]:.4f}')
print(f'Test Recall: {metrics["test_metrics"]["macro_recall"]:.4f}')

In [ ]:
# CELL 8: Prepare validator dataset
# Copy subset of PlantVillage for plant images
import shutil, random
from pathlib import Path

os.makedirs('data/validator/plant', exist_ok=True)
os.makedirs('data/validator/nonplant', exist_ok=True)

# Plant images from PlantVillage
all_imgs = list(Path('PlantVillage').rglob('*.jpg')) + list(Path('PlantVillage').rglob('*.JPG'))
sampled = random.sample(all_imgs, min(10000, len(all_imgs)))
for i, p in enumerate(sampled):
    shutil.copy(p, f'data/validator/plant/plant_{i:05d}.jpg')
print(f'Copied {len(sampled)} plant images')

# Non-plant: download from ImageNet subset
print('\nDownloading non-plant images from ImageNet-mini...')
!wget -q https://storage.googleapis.com/kagglestorage/imagenet_mini_nonplant.zip -O nonplant.zip
!unzip -q nonplant.zip -d data/validator/nonplant/ 2>/dev/null || echo 'Manual step: add 5000 non-plant images to data/validator/nonplant/'
print('Non-plant images:', len(list(Path('data/validator/nonplant').glob('*.jpg'))))

In [ ]:
# CELL 9: Train MobileNetV3 Validator
# Expected: ~30 minutes on T4
result = subprocess.run(['python', 'training/train_validator.py'], capture_output=True, text=True)
print(result.stdout[-2000:])
if result.returncode != 0:
    print('ERRORS:', result.stderr[-1000:])

In [ ]:
# CELL 10: Build RAG FAISS index
result = subprocess.run(['python', '-c', 'from services.rag_faiss import rag_pipeline; rag_pipeline.build_index()'], capture_output=True, text=True)
print(result.stdout)
if result.returncode != 0:
    print('ERRORS:', result.stderr)

In [ ]:
# CELL 11: Download trained models to your computer
from google.colab import files
for f in ['models/plant_classifier.pth', 'models/plant_validator.pth', 'models/plant_classes.json', 'models/plant_metrics.json', 'models/rag_faiss.index', 'models/rag_corpus.pkl', 'models/validator_metrics.json']:
    if os.path.exists(f):
        files.download(f)
        print(f'Downloaded: {f}')
    else:
        print(f'Missing: {f}')

In [ ]:
# CELL 12: Quick inference test
import torch
from pathlib import Path

ckpt = torch.load('models/plant_classifier.pth', map_location='cpu')
print('Classifier checkpoint keys:', list(ckpt.keys()))
print('Val F1:', ckpt['val_f1'])
print('Val Acc:', ckpt['val_acc'])
print('Classes:', len(ckpt['classes']), 'disease classes')
print('\nModel trained successfully and ready for deployment!')